# Shopping Concierge: Multi-Agent AI with Google ADK + Anthropic Claude

## What You Will Learn

- **Agent**: An AI assistant with an instruction (persona) that can answer questions
- **Tool**: A Python function the agent can call to search data or perform actions
- **Sub-agent**: A specialist agent that handles one task really well
- **Delegation**: A manager agent that routes work to the right specialist
- **Session memory**: The conversation remembers what you said earlier

By the end, you'll build a team of AI agents that help customers find gifts, apply coupons, and chat naturally.

## Step 0: Setup

Install the Google Agent Development Kit and LiteLlm (a translator that lets ADK talk to Anthropic's Claude).

In [1]:
# Install libraries (run once, takes ~30 seconds)
!pip install -q google-adk litellm


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


**Why LiteLlm?** ADK is built for Google's Gemini models. LiteLlm translates ADK's requests so they work with other AI providers like Anthropic.

Check that your Anthropic API key is available (LiteLlm reads it automatically from the environment):

In [2]:
import os
# Check if the API key exists in environment variables
key = os.environ.get("ANTHROPIC_API_KEY")
print("✓ API key found" if key else "✗ API key missing - set ANTHROPIC_API_KEY")

✓ API key found


Define the model once so we can reuse it for every agent. We're using **Claude Haiku 4.5**, Anthropic's fastest and cheapest model:

In [3]:
from google.adk.models.lite_llm import LiteLlm

# Define the model once - change this line to swap models everywhere
MODEL = LiteLlm(model="anthropic/claude-haiku-4-5-20251001")

Load mock product catalog and coupon codes (in a real app, this would come from a database):

In [4]:
# Mock product catalog - a list of dictionaries
# Each dictionary has keys: name, price (in rupees), category, recipient
CATALOG = [
    {"name": "Silk Scarf", "price": 1200, "category": "fashion", "recipient": "sister"},
    {"name": "Fountain Pen", "price": 2500, "category": "office", "recipient": "father"},
    {"name": "Yoga Mat", "price": 1800, "category": "fitness", "recipient": "mother"},
    {"name": "Bluetooth Speaker", "price": 3200, "category": "tech", "recipient": "brother"},
    {"name": "Cookbook", "price": 900, "category": "books", "recipient": "mother"},
    {"name": "Wallet", "price": 1500, "category": "fashion", "recipient": "father"},
    {"name": "Perfume", "price": 2800, "category": "beauty", "recipient": "sister"},
    {"name": "Coffee Mug Set", "price": 600, "category": "home", "recipient": "brother"},
]

# Mock coupon codes - a dictionary mapping code strings to discount percentages
COUPONS = {"FESTIVE10": 10, "SAVE20": 20, "WELCOME5": 5}

## Step 1: Your First Agent

An **agent** is an AI assistant with an **instruction** (its personality and job). Let's create a Stylist agent that gives gift advice.

We'll also make a tiny helper function `ask()` to talk to any agent.

### Understanding `async` and `await`

**Analogy**: Ordering food at a restaurant. You place an order (start a task) and the kitchen works *while you wait*. You don't freeze at the counter — you sit down and the waiter brings food when it's ready.

In Python, `async` marks a function that can "wait" for slow tasks (like calling an AI). `await` means "start this task and give me the result when ready."

**In Jupyter**: Just write `await` before the agent call. That's it!

In [ ]:
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner

# Create a Stylist agent with only an instruction (no tools yet)
stylist = Agent(
    name="fashion_stylist",
    model=MODEL,  # Use the model we defined earlier
    instruction="You are a fashion stylist who gives gift recommendations."
)

# Helper function: send a message to an agent and print the reply
async def ask(agent, text):
    # InMemoryRunner manages the conversation; run_debug sends one message
    runner = InMemoryRunner(agent=agent)
    events = await runner.run_debug(text, quiet=True)  # quiet=True hides debug output
    print(events[-1].content.parts[0].text)  # Print the agent's final reply

# Test the Stylist
await ask(stylist, "Suggest a gift for my sister who loves fashion")

## Step 2: Give the Agent a Tool

A **tool** is a Python function the agent can call. The AI reads the **docstring** (the text in triple quotes) and **parameter names** to decide when and how to use it.

Let's give a Product Search agent the ability to search our catalog:

In [ ]:
def search_catalog(max_price: int, keyword: str = ""):
    """Search the product catalog by price and optional keyword.
    
    Args:
        max_price: Maximum price in rupees
        keyword: Optional keyword to match in name, category, or recipient
    """
    # Filter products: price <= max_price AND keyword matches (if provided)
    results = [
        p for p in CATALOG 
        if p["price"] <= max_price and (
            not keyword or keyword.lower() in 
            f"{p['name']} {p['category']} {p['recipient']}".lower()
        )
    ]
    # Return a formatted string with rupee symbol
    return "\n".join([f"{p['name']}: ₹{p['price']}" for p in results])

# Create agent with the tool in a list
search_agent = Agent(
    name="product_search",
    model=MODEL,
    instruction="You help find products. Use search_catalog to answer queries.",
    tools=[search_catalog]  # Pass the function directly
)

await ask(search_agent, "Show gifts for sister under ₹2000")

## Step 3: A Second Specialist

Let's create a **Deals agent** that applies coupon codes. Each agent is a specialist at one job:

In [24]:
def apply_coupon(code: str, price: int):
    """Apply a coupon code to a price.
    
    Args:
        code: Coupon code (e.g. FESTIVE10)
        price: Original price in rupees
    """
    # Look up the discount percentage in the COUPONS dictionary
    discount = COUPONS.get(code.upper(), 0)  # Default to 0 if code not found
    final = price - (price * discount // 100)  # Calculate final price
    return f"₹{price} → ₹{final} ({discount}% off)" if discount else "Invalid code"

deals_agent = Agent(
    name="deals_specialist",
    model=MODEL,
    instruction="You apply coupon codes to prices.",
    tools=[apply_coupon]
)

await ask(deals_agent, "Apply FESTIVE10 to ₹2000")

Perfect! 🎉 

The coupon code **FESTIVE10** has been applied successfully!

**Result:** ₹2000 → **₹1800** (10% off)

You're saving ₹200 with this coupon!


## Step 4: Assemble the Team

Now create a **Concierge** (manager) agent with **sub-agents**. 

**Delegation**: The Concierge reads each specialist's `description` and routes work to the right one. You ask the Concierge, and it talks to specialists behind the scenes.

In [25]:
# Add descriptions to specialist agents (description helps the manager delegate)
search_agent.description = "Searches product catalog by price and keywords"
deals_agent.description = "Applies coupon codes to prices"

# Manager agent that delegates to specialists
concierge = Agent(
    name="concierge",
    model=MODEL,
    instruction="You are a shopping concierge. Help customers find gifts and deals.",
    sub_agents=[search_agent, deals_agent]  # List of specialist agents
)

# Complex query that needs both agents
await ask(concierge, "Find a gift for my sister under ₹3000. I have coupon FESTIVE10")

App "InMemoryRunner" can transfer between agents but has no context_cache_config. Every transfer swaps the system instruction and the tool set, so the request prefix changes and the whole prompt is re-sent uncached after each transfer. Set context_cache_config on the app to give each agent its own cache.


Excellent! Here are all your gift options with the **FESTIVE10 coupon applied** (10% off):

| Gift Item | Original Price | After FESTIVE10 |
|-----------|---|---|
| ☕ **Coffee Mug Set** | ₹600 | **₹540** |
| 📖 **Cookbook** | ₹900 | **₹810** |
| 🧣 **Silk Scarf** | ₹1200 | **₹1080** |
| 👛 **Wallet** | ₹1500 | **₹1350** |
| 🧘 **Yoga Mat** | ₹1800 | **₹1620** |
| ✒️ **Fountain Pen** | ₹2500 | **₹2250** |
| 💐 **Perfume** | ₹2800 | **₹2520** |

All items are still under ₹3000 after the discount! Here are my recommendations:

- **Best value**: Coffee Mug Set or Cookbook (great for gifting!)
- **Most elegant**: Silk Scarf or Perfume
- **Practical & useful**: Wallet or Yoga Mat
- **Luxury option**: Fountain Pen

Which one would you like to go with?


## Step 5: Follow-Up Conversation

A **session** remembers the conversation history. Let's send a follow-up message:

In [26]:
# Create a runner and send two messages (session remembers context)
runner = InMemoryRunner(agent=concierge)

# First message
events1 = await runner.run_debug("Show gifts for father under ₹2000", quiet=True)
print("👤 User: Show gifts for father under ₹2000")
print(f"🤖 Agent: {events1[-1].content.parts[0].text}\n")

# Follow-up (agent remembers we were talking about father's gifts)
events2 = await runner.run_debug("What if I use coupon SAVE20 on the pen?", quiet=True)
print("👤 User: What if I use coupon SAVE20 on the pen?")
print(f"🤖 Agent: {events2[-1].content.parts[0].text}")

👤 User: Show gifts for father under ₹2000
🤖 Agent: Great! Here are some gift options for your father under ₹2000:

1. **Coffee Mug Set** - ₹600
2. **Cookbook** - ₹900
3. **Silk Scarf** - ₹1200
4. **Wallet** - ₹1500
5. **Yoga Mat** - ₹1800

All of these are well within your budget! Popular choices for fathers would be the **Wallet** (practical and personal) or the **Coffee Mug Set** (if he enjoys his morning coffee). Would you like more information about any of these items or need help with anything else?

👤 User: What if I use coupon SAVE20 on the pen?
🤖 Agent: I don't see a pen in the search results I just showed you. The products available under ₹2000 were:

1. Coffee Mug Set - ₹600
2. Cookbook - ₹900
3. Silk Scarf - ₹1200
4. Wallet - ₹1500
5. Yoga Mat - ₹1800

Would you like me to:
1. Search for pens in a different price range?
2. Apply the coupon code SAVE20 to one of the products listed above?

Let me know and I can help you further!


## Recap

You built a **multi-agent system**:
1. **Agents** have instructions (personas)
2. **Tools** let agents call Python functions
3. **Sub-agents** are specialists at one task
4. **Delegation** routes work to the right specialist
5. **Sessions** remember conversation context

---

## Student Exercises

1. **Add a Returns agent**: Create `check_return_policy(days_since_purchase: int)` that returns "eligible" if ≤30 days, else "not eligible". Add it as a sub-agent.

2. **Change the persona**: Edit the Concierge instruction to make it talk like a pirate or a poet.

3. **Add a new coupon**: Add `"MEGA50": 50` to the `COUPONS` dict and test it.

**Tip**: Change the `MODEL` variable at the top to experiment with different AI models!